In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# =========================================
# LOAD DATA
# =========================================

train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

train_df.head()
# =========================================
# CHECK MISSING VALUES
# =========================================

missing = train_df.isnull().sum()

missing = missing[missing > 0]

missing.sort_values(ascending=False)
# =========================================
# DROP ID COLUMN
# =========================================

train_ids = train_df["Id"]
test_ids = test_df["Id"]

train_df.drop("Id", axis=1, inplace=True)
test_df.drop("Id", axis=1, inplace=True)
# =========================================
# HANDLE NUMERICAL MISSING VALUES
# =========================================

num_cols = train_df.select_dtypes(
    include=["int64", "float64"]
).columns

for col in num_cols:

    if train_df[col].isnull().sum() > 0:

        median_value = train_df[col].median()

        train_df[col] = train_df[col].fillna(
            median_value
        )

        if col in test_df.columns:

            test_df[col] = test_df[col].fillna(
                median_value
            )

print("Numerical missing values handled.")
# =========================================
# SPECIAL DOMAIN CLEANING
# =========================================

special_cols = [
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "PoolQC",
    "Fence",
    "MiscFeature",
    "Alley"
]

for col in special_cols:

    if col in train_df.columns:

        train_df[col] = train_df[col].fillna(
            "None"
        )

    if col in test_df.columns:

        test_df[col] = test_df[col].fillna(
            "None"
        )

print("Special columns cleaned.")
# =========================================
# HANDLE REMAINING CATEGORICAL MISSING VALUES
# =========================================

cat_cols = train_df.select_dtypes(
    include=["object"]
).columns

for col in cat_cols:

    train_df[col] = train_df[col].astype(str)

    if col in test_df.columns:
        test_df[col] = test_df[col].astype(str)

    if train_df[col].isnull().sum() > 0:

        mode_value = train_df[col].mode()[0]

        train_df[col] = train_df[col].fillna(
            mode_value
        )

        if col in test_df.columns:

            test_df[col] = test_df[col].fillna(
                mode_value
            )

print("Remaining categorical missing values handled.")
# =========================================
# REMOVE OUTLIERS
# =========================================

train_df = train_df[
    ~(
        (train_df["GrLivArea"] > 4000) &
        (train_df["SalePrice"] < 300000)
    )
]

print("Outliers removed.")
# =========================================
# SAVE CLEANED DATA
# =========================================

train_df.to_csv(
    "../data/processed/cleaned_train.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/cleaned_test.csv",
    index=False
)

print("Cleaned datasets saved.")

Train Shape: (1168, 81)
Test Shape : (292, 80)
Numerical missing values handled.
Special columns cleaned.
Remaining categorical missing values handled.
Outliers removed.
Cleaned datasets saved.
